# B站：关键词搜索采集

按关键词发现视频，再采集该视频实际返回或展示的评论。网络评论需用视频aid核对；公开shadow DOM未暴露原生rpid时，comment_id保留空值并标missing。

默认只做路径和参数检查，不联网采集。把 RUN_CRAWLER 改为 True 后，在有桌面的本机 Jupyter 运行；会打开独立临时浏览器。遇到登录/验证码，先自行处理，回到Notebook输入 r 继续或 q 停止。关闭浏览器后不会保存登录状态。

需要Python 3.10或以上。首次运行若缺依赖，在当前内核执行 `%pip install playwright tzdata`，然后用该内核的 Python 执行 `-m playwright install chromium`；也可把 BROWSER_CHANNEL 改为 'chrome' 或 'msedge'，使用本机已安装浏览器。代码不自动安装软件、不更改浏览器安全设置。Playwright用单独线程运行，兼容Notebook已有事件循环。

输出固定在 `../数据/01_原始数据/B站/new/`。CSV/JSONL是正文数据，SQLite是事务检查点；同一搜索词和窗口参数重复运行会去重并跳过已结束主题窗口。改变窗口参数会产生另一组文件。旧版原始文件不改动，新采集数据不会自动进入分析。

交付测试状态：已在真实网页核验本平台的关键词搜索与评论/正文DOM字段，并通过离线程序测试；普通Python新浏览器从启动、手动登录到导出的端到端流程，仍因云端运行环境限制未完成验收。真实页面测试会话不会打包或复用。


In [27]:
from pathlib import Path
import importlib.util
import sys

# 在 lasso-新数据/python 或 lasso-新数据 目录启动Jupyter均可
_candidates = [Path.cwd(), Path.cwd() / "python", Path.cwd().parent / "python"]
_helper_dir = next((p for p in _candidates if (p / "crawler_helpers.py").is_file()), None)
if _helper_dir is None:
    raise FileNotFoundError("请从 lasso-新数据 或 lasso-新数据/python 目录启动Jupyter，并保留crawler_helpers.py")
_helper_spec = importlib.util.spec_from_file_location("lasso1016_crawler_helpers", _helper_dir / "crawler_helpers.py")
_helper = importlib.util.module_from_spec(_helper_spec)
sys.modules[_helper_spec.name] = _helper
_helper_spec.loader.exec_module(_helper)
CrawlConfig, dependency_check = _helper.CrawlConfig, _helper.dependency_check
output_dir, project_root, run_collection = _helper.output_dir, _helper.project_root, _helper.run_collection
ROOT = project_root(_helper_dir.parent)
print("项目目录：", ROOT.name)
print(dependency_check())


项目目录： lasso-1016
{'playwright_installed': True, 'version': '1.64.0', 'next_step': "首次使用需安装Chromium：python -m playwright install chromium；或BROWSER_CHANNEL='chrome'使用已安装Chrome"}


In [67]:
# 示例，可替换为你的搜索词；不是最终采样清单，不做Y/情感倾向筛选
QUERIES = ["抖音双11"]
TOPIC_PUBLISHED_YEAR = 2023  # B站“更多筛选”的视频发表年份；评论时间不据此筛选
TOPIC_PUBLISHED_START = f"{TOPIC_PUBLISHED_YEAR}-01-01"
TOPIC_PUBLISHED_END = f"{TOPIC_PUBLISHED_YEAR}-12-31"
PLATFORM = "B站"
RUN_CRAWLER = True        # 手动改为True才会启动真实采集
MAX_TOPICS = 5             # 每次最多访问的主题数；微博为最多帖子数
MAX_COMMENTS_PER_TOPIC = 30
SEARCH_SCROLLS = 3         # 搜索页额外滚动窗口数
COMMENT_SCROLLS = 6        # 每个主题额外评论滚动窗口数
SEARCH_PAGES = 2           # 微博搜索页数；其他平台采用滚动窗口
PAUSE_SECONDS = 3.0        # 每次导航/滚动之间至少3秒
MAX_RUNTIME_MINUTES = 20   # 有限运行时间；手动处理也计时
BROWSER_CHANNEL = "chrome"  # 使用已安装的本机 Chrome；避免依赖缺失的 Playwright Chromium

config = CrawlConfig(
    platform=PLATFORM, queries=QUERIES, root=str(ROOT),
    max_topics=MAX_TOPICS, max_comments_per_topic=MAX_COMMENTS_PER_TOPIC,
    search_scrolls=SEARCH_SCROLLS, comment_scrolls=COMMENT_SCROLLS,
    search_pages=SEARCH_PAGES, pause_seconds=PAUSE_SECONDS,
    max_runtime_minutes=MAX_RUNTIME_MINUTES,
    browser_channel=BROWSER_CHANNEL, interactive=True,
    topic_published_start=TOPIC_PUBLISHED_START, topic_published_end=TOPIC_PUBLISHED_END,
)
config.validate()
OUTPUT_DIR = output_dir(PLATFORM, ROOT)
print("B站搜索发布时间筛选：", TOPIC_PUBLISHED_START, "至", TOPIC_PUBLISHED_END)
print("新采集输出：", OUTPUT_DIR.relative_to(ROOT))


B站搜索发布时间筛选： 2023-01-01 至 2023-12-31
新采集输出： 数据/01_原始数据/B站/new


In [ ]:
if RUN_CRAWLER:
    reports = run_collection(config)
else:
    reports = []
    print("已完成参数与路径检查；未联网、未启动采集。需要采集时，将RUN_CRAWLER改为True并重跑。")


B站｜搜索词：抖音双11


未读到可核验关联的评论。若评论折叠，请手动打开评论区；若有登录或验证码，请自行处理。请在刚打开的独立浏览器中手动处理；完成后输入 r，停止输入 q： r


主题 BV1hH4y1B7SZ：没有可核验评论，保留未完成状态
主题 BV1sa4y1Q7by：累计保存 30 条
主题 BV1rQ4y187GS：累计保存 19 条
主题 BV1dC4y177JQ：累计保存 30 条
主题 BV1ka4y1S7Pz：累计保存 30 条
主题 BV1dG411Q7rk：累计保存 11 条
抖音双11：partial，累计 120 条
有主题没有捕获到可核验评论，不能解释为真正零评论；可重新运行未完成主题
CSV： /Users/xuxiaoxue/Documents/Writings/DA-holiday-work/reasons-analysis-of-shopping-and-rules/lasso-1016/数据/01_原始数据/B站/new/抖音双11_b293ce625a5d.csv


In [4]:
# 只查看本Notebook的new输出，不读取或改动旧版分析数据
import csv
for report in reports:
    path = report.get("csv")
    if not path or not Path(path).is_file():
        continue
    with Path(path).open(encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    print(Path(path).name, "行数：", len(rows))
    print("缺少原生评论ID：", sum(r["comment_id_status"] == "missing" for r in rows))
    print("评论/正文时间不可解析：", sum(not r["comment_time_iso"] for r in rows))
    print("主题时间不可解析：", sum(not r["topic_time_iso"] for r in rows))
    print("运行状态：", report["status"])


时间原值保存在 topic_time_raw / comment_time_raw，能解析的绝对时间另存 *_iso。相对时间注明 relative_estimate / relative_day_inferred；缺少年份的“月-日”保留空值并标 missing_year，不猜年份。正文、主题标题/内容、topic_id/URL、来源平台、抓取时间和搜索词均保留。

comment_id_status=native 才表示平台原生评论ID。local_record_id始终是本地记录ID；无原生ID时，按正文、时间原值和同窗口重复序号去重，可能合并跨窗口不可区分的同文评论，不声称精确识别全部评论。仅采有限可见/已加载窗口，不自动展开全部楼中楼，不保证全量。没有读到评论会标未完成，不把登录失败解释为零评论。若页面结构变更，可能需要调整辅助模块；不要把空输出当成采集成功。
